# Chain Rule and Computation Graphs

| Difficulty | ████░░░░░░ 4/10 |
| :--- | :--- |
| Prerequisites | 02.03 |
| Time | ~70 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/02_Calculus/04_chain_rule_and_computation_graphs.ipynb)

---

## 🎯 Learning Objective

Represent computations as directed acyclic graphs and derive forward-mode and reverse-mode differentiation from the multivariate chain rule.

---

## 📐 Theory

This is the single most important notebook in Module 02: backpropagation, the algorithm that
trains every neural network, is nothing more than the chain rule applied systematically.

### The univariate chain rule, again

From `02.02`: if $y=f(u)$ and $u=g(x)$, then $\frac{dy}{dx} = \frac{dy}{du}\cdot\frac{du}{dx}$.
Composing functions multiplies their local rates of change. Think of every computation as a
**chain of simple steps**, each with an easy local derivative, and let the chain rule stitch
the local pieces into the global derivative.

### The multivariate chain rule

When an intermediate quantity feeds multiple downstream paths, each path contributes:

$$\frac{\partial z}{\partial x} = \sum_{i} \frac{\partial z}{\partial u_i}\cdot\frac{\partial u_i}{\partial x}$$

if $x$ influences $z$ through several intermediates $u_1,\dots,u_k$. This "sum over paths" is
exactly what makes the computation-graph picture below necessary — a variable used twice must
have its influence counted twice.

### Computation graphs

Any composite expression is a **directed acyclic graph (DAG)**: nodes are variables (inputs, or
operation outputs), edges point from inputs to the operation consuming them. $z=(x+y)\cdot x$
decomposes as $a=x+y$, $z=a\cdot x$ — a graph where $x$ feeds both the $+$ node and the
$\times$ node (a "fan-out"), precisely the case the sum-over-paths rule handles.

### Forward-mode vs. reverse-mode differentiation

**Forward mode** propagates derivatives alongside values, in the same order as the original
computation: for a fixed input $x$, compute $\partial(\text{node})/\partial x$ from its
parents' values. This gets the derivative of every output w.r.t. *one* input per pass.

**Reverse mode** (backpropagation) runs the other way: a full forward pass computes every
node's value, then a backward walk computes $\partial(\text{output})/\partial(\text{node})$
using the cached values. This gets the derivative of *one* output w.r.t. *every* input in a
single pass — exactly a loss function's shape (millions of parameters, one scalar loss). This
asymmetry is why every deep learning framework uses reverse mode: its cost is roughly one extra
forward pass, regardless of parameter count (measured directly in `02.03`).

### The local-gradient recipe

Reverse mode needs one thing per operation: its **local gradient**. Multiplication $z=ab$ has
local gradients $\partial z/\partial a=b$, $\partial z/\partial b=a$; addition $z=a+b$ has
local gradients $1,1$. Backprop then multiplies the incoming "upstream gradient" by each local
gradient and routes the result backward along every edge, *summing* at fan-out nodes. Chain
enough of these together and you can differentiate an arbitrarily deep expression without ever
writing its derivative symbolically — exactly what we build below.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Drawing $z = (x + y) \cdot x$ as a DAG makes the fan-out case (where $x$ feeds two downstream
nodes) visible: the forward pass flows left-to-right computing values, and the backward pass
(arrows reversed) flows right-to-left accumulating gradients, summing wherever a node forked.

In [ ]:
# Draw the computation graph for z=(x+y)*x, with forward values (nodes) and local
# gradients (edges) -- via networkx/matplotlib, no external 'dot' binary needed.
x_val, y_val = 3.0, 4.0
a_val = x_val + y_val
z_val = a_val * x_val
dz_da, dz_dx_direct = x_val, a_val          # local gradients of z=a*x
dz_dx_via_add = dz_da * 1.0                  # chained through a=x+y
dz_dx_total = dz_dx_direct + dz_dx_via_add   # FAN-OUT: sum the two paths x contributed through

G = nx.DiGraph()
G.add_edges_from([("x", "a"), ("y", "a"), ("a", "z"), ("x", "z")])
pos = {"x": (0, 0), "y": (0, 2), "a": (1.5, 1), "z": (3, 1)}
fig, ax = plt.subplots(figsize=(9, 5))
nx.draw_networkx_nodes(G, pos, node_size=1800, node_color="#4C72B0", alpha=0.85, ax=ax)
nx.draw_networkx_edges(G, pos, arrowsize=20, node_size=1800, ax=ax)
labels = {"x": f"x={x_val}", "y": f"y={y_val}", "a": f"a=x+y\n={a_val}", "z": f"z=a*x\n={z_val}"}
nx.draw_networkx_labels(G, pos, labels, font_color="white", font_size=9, ax=ax)
edge_grads = {("x", "a"): "1", ("y", "a"): "1", ("a", "z"): f"{x_val}", ("x", "z"): f"{a_val}"}
for (u, v), label in edge_grads.items():
    mx, my = (pos[u][0] + pos[v][0]) / 2, (pos[u][1] + pos[v][1]) / 2
    ax.annotate(f"local grad={label}", (mx, my), fontsize=8, color="#C44E52", ha="center")
ax.set_title("Forward values (nodes) + local gradients (edges) for z=(x+y)*x")
ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Total dz/dx = (direct path) {dz_dx_direct} + (via a) {dz_dx_via_add} = {dz_dx_total}")
print(f"Analytic check: z=(x+y)x=x^2+xy, so dz/dx=2x+y = {2*x_val + y_val}")

## 🐍 Implementation from Scratch

We build a minimal reverse-mode autodiff engine: a `Value` class that records every operation
it's used in, then a `backward()` method that walks the recorded graph in reverse, applying
the local-gradient recipe from the Theory section at every step. This is a miniature version of
exactly what `torch.autograd` does internally.

In [ ]:
class Value:
    """A scalar that remembers how it was computed, so gradients can flow backward through it --
    the smallest possible version of what torch.Tensor does under the hood."""
    def __init__(self, data, _parents=(), _local_grads=()):
        self.data = data
        self.grad = 0.0
        self._parents = _parents          # Value objects that produced this one
        self._local_grads = _local_grads  # d(self)/d(parent), for each parent, in order

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), (1.0, 1.0))  # d(a+b)=1,1

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), (other.data, self.data))  # d(ab)=b,a

    def tanh(self):
        t = np.tanh(self.data)
        return Value(t, (self,), (1 - t**2,))  # d(tanh(a))/da = 1 - tanh(a)^2

    def backward(self):
        order, visited = [], set()
        def build(v):  # topological sort: process a node only after all its consumers
            if id(v) not in visited:
                visited.add(id(v))
                for p in v._parents:
                    build(p)
                order.append(v)
        build(self)
        self.grad = 1.0  # dz/dz = 1, the seed of every reverse-mode pass
        for v in reversed(order):
            for parent, local_grad in zip(v._parents, v._local_grads):
                parent.grad += v.grad * local_grad  # chain rule; += handles fan-out

# Reproduce z = (x+y)*x from the Visual Intuition cell
x, y = Value(3.0), Value(4.0)
z = (x + y) * x
z.backward()
print(f"z.data={z.data} (expect 21), x.grad={x.grad} (expect 2x+y=10), y.grad={y.grad} (expect x=3)")

# A deeper graph with a nonlinearity: a tiny 1-neuron "network"
w1, w2, b = Value(0.5), Value(-0.3), Value(0.1)
out = (w1 * Value(2.0) + w2 * Value(1.0) + b).tanh()
out.backward()
print(f"\nNeuron output = tanh(w1*x1+w2*x2+b) = {out.data:.5f}")
print(f"d(out)/d(w1)={w1.grad:.5f}, d(out)/d(w2)={w2.grad:.5f}, d(out)/d(b)={b.grad:.5f}")

# Cross-check against PyTorch's autograd -- a correct from-scratch engine must match exactly
import torch
w1_t, w2_t, b_t = (torch.tensor(v, requires_grad=True) for v in (0.5, -0.3, 0.1))
torch.tanh(w1_t * 2.0 + w2_t * 1.0 + b_t).backward()
print(f"PyTorch cross-check -- d(out)/d(w1): {w1_t.grad.item():.5f} (ours: {w1.grad:.5f})")

## 🤖 Why This Matters for AI

`torch.autograd`, `tf.GradientTape`, and JAX's `grad` are all reverse-mode automatic
differentiation engines — larger and vectorized, but built from exactly the `Value` class
above: every tensor op records its inputs and local gradient rule, and `.backward()` walks the
graph in reverse. **Backpropagation is not separate from the chain rule** — it *is* the
multivariate chain rule via reverse-mode differentiation over the DAG a forward pass traces
out. Every weight is a leaf feeding the scalar loss at the root, and `.backward()` computes
every leaf's gradient in one traversal. Below, a tiny 2-layer network built entirely from our
`Value` class (no `torch.nn`) learns a toy task with plain gradient descent.

In [ ]:
# Training a tiny 2-layer network using ONLY our from-scratch Value/backward() engine --
# no torch.nn, no autograd library -- to prove reverse-mode chain rule is all learning needs.
rng = np.random.default_rng(0)

class Neuron:
    def __init__(self, n_in):
        self.w = [Value(rng.normal(scale=0.5)) for _ in range(n_in)]
        self.b = Value(0.0)
    def __call__(self, xs):
        act = self.b
        for wi, xi in zip(self.w, xs):
            act = act + wi * xi
        return act.tanh()
    def parameters(self):
        return self.w + [self.b]

class MLP:
    def __init__(self, n_in, n_hidden):
        self.hidden = [Neuron(n_in) for _ in range(n_hidden)]
        self.output = Neuron(n_hidden)
    def __call__(self, xs):
        return self.output([neuron(xs) for neuron in self.hidden])
    def parameters(self):
        return [p for n in self.hidden for p in n.parameters()] + self.output.parameters()

# Toy task: learn an XOR-like boundary -- impossible for a single linear neuron, which is
# exactly why the chain rule through a HIDDEN layer matters
X_toy = [(0.0, 0.0), (0.0, 1.0), (1.0, 0.0), (1.0, 1.0)]
y_toy = [0.0, 1.0, 1.0, 0.0]
net = MLP(n_in=2, n_hidden=4)
losses, lr = [], 0.3
for step in range(200):
    total_loss = Value(0.0)
    for xi, yi in zip(X_toy, y_toy):
        diff = net([Value(xi[0]), Value(xi[1])]) + Value(-yi)
        total_loss = total_loss + diff * diff  # squared error, via our own __add__/__mul__
    for p in net.parameters():
        p.grad = 0.0            # reset gradients, same as optimizer.zero_grad()
    total_loss.backward()        # ONE call computes gradients for every parameter in the net
    for p in net.parameters():
        p.data -= lr * p.grad    # plain gradient descent -- no momentum/Adam needed here
    losses.append(total_loss.data)

print(f"Loss: {losses[0]:.4f} -> {losses[-1]:.4f} (learned via chain rule alone, zero library autograd)")
for xi, yi in zip(X_toy, y_toy):
    pred = net([Value(xi[0]), Value(xi[1])])
    print(f"  input={xi}  target={yi}  predicted={pred.data:.3f}")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(losses, color="#4C72B0", lw=2)
ax.set_xlabel("training step"); ax.set_ylabel("total squared error")
ax.set_title("A network trained with our from-scratch Value.backward() -- pure chain rule")
plt.show()

## 🏋️ Exercises

### Warm-up
1. By hand, apply the multivariate chain rule to $z=(x+y)\cdot(x-y)$ to find $\partial z/\partial x$
   and $\partial z/\partial y$ at $(x,y)=(3,1)$. Verify with `Value` and `.backward()`.

### Practice
2. Add a `__pow__` method to `Value` supporting `v ** n` for a constant integer $n$, with local
   gradient $n\cdot v^{n-1}$ (power rule, `02.02`). Test on $f(x)=x^3$ at $x=2$, confirm
   `x.grad` equals $3x^2=12$.

### Challenge
3. The `MLP` above stores every number as a scalar `Value` object rather than a tensor. Add a
   class-level counter to `Value.__init__` and report how many `Value` objects one `net(xs)`
   call creates for `n_hidden=4`. Explain what would happen to that count if `n_hidden` were
   4096 (a real transformer hidden size), and why frameworks batch numbers into tensors instead.

---

## 📚 Further Reading
- Karpathy, [micrograd](https://github.com/karpathy/micrograd) — direct inspiration for this notebook's `Value` engine
- Baydin et al., ["Automatic Differentiation in Machine Learning: a Survey"](https://arxiv.org/abs/1502.05767)
- Rumelhart, Hinton & Williams, ["Learning representations by back-propagating errors"](https://www.nature.com/articles/323533a0) (1986)

---

*Next notebook: [Gradients and Directional Derivatives](05_gradients_and_directional_derivatives.ipynb)*